<a href="https://colab.research.google.com/github/KarthikRed2000/grounded_vla/blob/main/colab/01_setup_drive_and_weights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01: Drive Setup + Download Model Weights

**Run once.** Mounts Google Drive and downloads LLaVA-1.6-7B + Mistral-7B-Instruct
into `/content/drive/MyDrive/grounded_vla/hf_cache`. Subsequent notebooks read
weights from Drive without re-downloading.

## Prerequisites

1. Accept the model licenses on HuggingFace:
   - https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf
   - https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2
2. Add a `HF_TOKEN` secret in Colab (left sidebar → key icon).
3. Runtime → Change runtime type → **A100 GPU** (or T4 if A100 unavailable).
4. Make sure your Drive has **~30 GB free**.

**Approximate cost:** ~10 min runtime, ~1 compute unit on A100.

In [9]:
# Mount Google Drive. The first run prompts for OAuth; subsequent runs
# in the same session reuse the token automatically.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/grounded_vla')
ROOT.mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive root: /content/drive/MyDrive/grounded_vla


In [2]:
# Verify accelerator. Pro+ should give you A100 most of the time; if you
# see T4, change Runtime -> Change runtime type -> A100 GPU and re-run.
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv'])

CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], returncode=0)

In [3]:
# Disk check on Drive (Colab caches Drive lazily, so 'free' here is the
# Colab side; actual Drive free space is in your Google account quota).
import shutil
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'/content/drive/MyDrive: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

/content/drive/MyDrive: 34.6 GB free / 107.4 GB total


In [4]:
!pip install -q --upgrade huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 11.8 MB/s eta 0:00:00


In [5]:
# Authenticate with HF using a token stored as a Colab Secret.
# Add the secret first: left sidebar -> key icon -> 'HF_TOKEN' -> paste token,
# then toggle 'Notebook access' on for this notebook.
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
print('HF auth OK')

HF auth OK


In [6]:
from huggingface_hub import snapshot_download
from pathlib import Path

CACHE = Path('/content/drive/MyDrive/grounded_vla/hf_cache')
CACHE.mkdir(parents=True, exist_ok=True)

print('Downloading LLaVA-1.6-7B (~13 GB) ...')
snapshot_download(
    repo_id='llava-hf/llava-v1.6-mistral-7b-hf',
    local_dir=CACHE / 'llava-v1.6-mistral-7b-hf',
    local_dir_use_symlinks=False,
    ignore_patterns=['*.bin', '*.msgpack', '*.h5'],
)
print('LLaVA done.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

LLaVA done.


In [7]:
print('Downloading Mistral-7B-Instruct (~14 GB) ...')
snapshot_download(
    repo_id='mistralai/Mistral-7B-Instruct-v0.2',
    local_dir=CACHE / 'Mistral-7B-Instruct-v0.2',
    local_dir_use_symlinks=False,
    ignore_patterns=['*.bin', '*.msgpack', '*.h5'],
)
print('Mistral done.')

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Mistral done.


In [8]:
import subprocess
subprocess.run(['du', '-sh', '/content/drive/MyDrive/grounded_vla/hf_cache'])
for sub in sorted(CACHE.iterdir()):
    print(sub.name)

Mistral-7B-Instruct-v0.2
llava-v1.6-mistral-7b-hf


## Done

Weights are now permanently in your Drive. **You don't need to re-run this**
unless model versions change. Move on to `02_setup_data.ipynb`.

Tip: if Drive sync is slow afterwards, you can disable Drive auto-sync on the
`hf_cache` folder via the Drive UI (right-click → 'Available offline' → off).